# Olist E-Commerce Data Cleaning

This notebook covers the cleaning of five core CSV files from the Olist Brazilian E-Commerce dataset.
The goal is to produce clean, analysis-ready files that will be used across SQL, Excel, and Power BI
in the subsequent stages of this project. Each file is first passed through a common cleaning function
to handle duplicates and column formatting, and then cleaned individually based on its own specific issues.

In [31]:
import pandas as pd 
import numpy as np 
import os

In [32]:
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')

In [33]:
def basic_clean(df):
    df.drop_duplicates(inplace=True)
    df.columns = df.columns.str.strip().str.lower()
    return df

df_orders = basic_clean(df_orders)
df_items = basic_clean(df_items)
df_payments = basic_clean(df_payments)
df_customers = basic_clean(df_customers)
df_reviews = basic_clean(df_reviews)

In [34]:
def null_values_check(df, name):
    print(f'\nNull values in {name}:')
    print(df.isna().sum())
    return df

df_orders = null_values_check(df_orders, 'orders')
df_items = null_values_check(df_items, 'items')
df_payments = null_values_check(df_payments, 'payments')
df_customers = null_values_check(df_customers, 'customers')
df_reviews = null_values_check(df_reviews, 'reviews')


Null values in orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Null values in items:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Null values in payments:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Null values in customers:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Null values in reviews:
review_id                      0
order_id                       

In [35]:
print(f'before dropping values in reviews csv: {df_reviews.shape}')
df_reviews.drop(columns=['review_comment_title', 'review_comment_message'], inplace=True)
print(f'After dropping values in reviews csv: {df_reviews.shape}')

before dropping values in reviews csv: (99224, 7)
After dropping values in reviews csv: (99224, 5)


## Handling Null Values

After inspecting all five dataframes, null values were found in two files: orders and reviews.

In the orders file, three columns contained nulls. These were order_approved_at, order_delivered_carrier_date,
and order_delivered_customer_date. These nulls are not data quality issues. They exist because some orders
were cancelled before approval, and some orders had not yet been shipped or delivered at the time the data
was collected. Filling or dropping these rows would remove valid order records and distort any time-based
analysis. These columns were left as NaT.

In the reviews file, two columns were dropped entirely. The review_comment_title column had 87,656 null
values and review_comment_message had 58,247 null values. Both columns are optional text fields that the
majority of customers did not fill in. They carry no measurable analytical value for business questions
and were therefore removed from the dataframe.

The remaining three files, items, payments, and customers, contained no null values and required no action.

In [36]:
for name, df in [('orders', df_orders), ('items', df_items), ('payments', df_payments), 
                  ('customers', df_customers), ('reviews', df_reviews)]:
    print(f'\nDtypes in {name}:')
    print(df.dtypes)


Dtypes in orders:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Dtypes in items:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Dtypes in payments:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

Dtypes in customers:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype:

In [37]:
# Orders, fixing the timestamps
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in timestamp_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

# Items, fixing shipping date
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])

# Customers, zip code to string
df_customers['customer_zip_code_prefix'] = df_customers['customer_zip_code_prefix'].astype(str)

# Reviews, fixing the timestamps
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'])
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'])

## Fixing Data Types

After inspecting the dtypes across all five dataframes, several columns were stored as object
when they should have been datetime, and one column was stored as a numeric type when it should
have been a string.

In the orders dataframe, all five timestamp columns were stored as object. These were
order_purchase_timestamp, order_approved_at, order_delivered_carrier_date,
order_delivered_customer_date, and order_estimated_delivery_date. All five were converted
to datetime using pd.to_datetime().

In the items dataframe, the shipping_limit_date column was also stored as object and was
converted to datetime.

In the customers dataframe, customer_zip_code_prefix was stored as int64. Zip codes are
identifiers and not numbers you perform arithmetic on, so this column was converted to string.

In the reviews dataframe, review_creation_date and review_answer_timestamp were both stored
as object and were converted to datetime.

The payments dataframe required no dtype changes.

In [38]:
for name, df in [('orders', df_orders), ('items', df_items), ('payments', df_payments),
                  ('customers', df_customers), ('reviews', df_reviews)]:
    print(f'\nDuplicates in {name}: {df.duplicated().sum()}')


Duplicates in orders: 0

Duplicates in items: 0

Duplicates in payments: 0

Duplicates in customers: 0

Duplicates in reviews: 0


In [39]:
for name, df in [('orders', df_orders), ('items', df_items), ('payments', df_payments),
                  ('customers', df_customers), ('reviews', df_reviews)]:
    print(f'\nString columns in {name}:')
    for col in df.select_dtypes(include='object').columns:
        print(f'  {col}: {df[col].unique()[:5]}')


String columns in orders:
  order_id: ['e481f51cbdc54678b7cc49136f2d6af7' '53cdb2fc8bc7dce0b6741e2150273451'
 '47770eb9100c2d0c44946d9cf07ec65d' '949d5b44dbf5de918fe9c16f97b45f8a'
 'ad21c59c0840e6cb83a9ceb5573f8159']
  customer_id: ['9ef432eb6251297304e76186b10a928d' 'b0830fb4747a6c6d20dea0b8c802d7ef'
 '41ce2a54c0b03bf3443c3d931a367089' 'f88197465ea7920adcdbec7375364d82'
 '8ab97904e6daea8866dbdbc4fb7aad2c']
  order_status: ['delivered' 'invoiced' 'shipped' 'processing' 'unavailable']

String columns in items:
  order_id: ['00010242fe8c5a6d1ba2dd792cb16214' '00018f77f2f0320c557190d7a144bdd3'
 '000229ec398224ef6ca0657da4fc703e' '00024acbcdf0a6daa1e931b038114c75'
 '00042b26cf59d7ce69dfabb4e55b4fd9']
  product_id: ['4244733e06e7ecb4970a6e2683c13e61' 'e5f2d52b802189ee658865ca93d83a8f'
 'c777355d18b72b67abbeef9df44fd0fd' '7634da152a4610f1595efa32f14722fc'
 'ac6c3623068f30de03045865e4e10089']
  seller_id: ['48436dade18ac8b2bce089ec2a041202' 'dd7ddc04e1b6c2c614352b383efe2d36'
 '5b51032eddd242

In [40]:
# Payments, replacing not_defined
df_payments['payment_type'] = df_payments['payment_type'].replace('not_defined', 'Unknown')

# Customers, fixing the city casing
df_customers['customer_city'] = df_customers['customer_city'].str.title()

# verify 
print(df_payments['payment_type'].unique())
print(df_customers['customer_city'].unique()[:5])

['credit_card' 'boleto' 'voucher' 'debit_card' 'Unknown']
['Franca' 'Sao Bernardo Do Campo' 'Sao Paulo' 'Mogi Das Cruzes' 'Campinas']


## Fixing String Inconsistencies

After inspecting all string columns across the five dataframes, two issues were found.

In the payments dataframe, the payment_type column contained a value labeled not_defined.
This is not a valid payment category and carries no analytical meaning. It was replaced
with Unknown to make it explicit that the payment method was not recorded for those orders.

In the customers dataframe, the customer_city column was entirely lowercase. City names
should be in title case for readability and consistency. The column was converted using
str.title() which correctly formatted values like franca to Franca and sao paulo to Sao Paulo.

No string issues were found in the orders, items, or reviews dataframes.

In [41]:
for name, df in [('orders', df_orders), ('items', df_items), ('payments', df_payments),
                  ('customers', df_customers), ('reviews', df_reviews)]:
    print(f'\n{name}: {list(df.columns)}')


orders: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

items: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

payments: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

customers: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

reviews: ['review_id', 'order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']


In [42]:
files = {
    'orders_cleaned.csv': df_orders,
    'items_cleaned.csv': df_items,
    'payments_cleaned.csv': df_payments,
    'customers_cleaned.csv': df_customers,
    'reviews_cleaned.csv': df_reviews
}

for filename, df in files.items():
    try:
        df.to_csv(filename, index=False)
        print(f'{filename} exported successfully')
    except Exception as e:
        print(f'Failed to export {filename}: {e}')

orders_cleaned.csv exported successfully
items_cleaned.csv exported successfully
payments_cleaned.csv exported successfully
customers_cleaned.csv exported successfully
reviews_cleaned.csv exported successfully


## Conclusion

The cleaning process across all five Olist dataframes is now complete. Duplicates were removed,
column names were standardized, data types were corrected, string inconsistencies were fixed,
and null values were handled with deliberate decisions rather than blanket dropping.

The five cleaned files have been exported as CSV and will serve as the single source of truth
for the remaining stages of this project including SQL analysis, Excel reporting, and the
final Power BI dashboard.

The raw data was never modified. All cleaning was performed on loaded dataframes and exported
as separate cleaned files.